# 03 - Support Vector Machine, RBF kernel (BEST MODEL)

This is our strongest model. After tuning every model family and three ensemble
strategies, a single RBF-SVM was best - and the one change that pushed it clearly
ahead was **how we fill the 50%-missing `eog_burst_index` column** (Section 6).

## 1. What this model does
A linear SVM finds the boundary that separates classes with the widest margin.
Our classes are not linearly separable, so the **RBF kernel** measures how close
each pair of points is (a Gaussian of width `gamma`), which bends the boundary
into smooth curves. For 4 classes, scikit-learn trains several one-vs-one SVMs and votes.

- **`C`** - error cost. Higher = punish training errors harder (wigglier boundary).
- **`gamma`** - kernel reach. Higher = each point influences a smaller neighbourhood.

## 2. Why it fits this data
- The stages overlap in curved, nonlinear regions; RBF models exactly that.
- Robust to the strong feature correlations (up to 0.82) found in EDA.

## 3. Costs / weaknesses
- Scale-sensitive (kernel uses distance) -> must standardize.
- No native NaN handling -> the missing column must be imputed (and HOW we impute
  turned out to matter a lot - see Section 6).
- ~O(n^2) training; fine at 9000 rows.

## 4. Performance summary (repeated 5-fold CV, macro-F1)
| Version | CV macro-F1 |
|---|---|
| baseline `C=3, gamma='scale'`, median-impute | 0.829 |
| tuned `C=10, gamma=0.01`, median-impute | 0.832 |
| **tuned `C=12, gamma=0.012` + ITERATIVE impute** | **0.842** |

The jump from 0.832 -> 0.842 came from the imputation change, validated as
statistically significant (paired t-test p ~ 5e-11, improved in 28/30 folds).

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd

TRAIN_PATH, TEST_PATH, OUT_DIR = '../train.csv', '../test.csv', '../outputs'
train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
FEATURES = [c for c in train.columns if c not in ('id', 'sleep_stage')]
TARGET, y = 'sleep_stage', None
y = train[TARGET]
print('train', train.shape, '| test', test.shape)
print('eog_burst_index missing: train %.0f%%, test %.0f%%'
      % (100*train.eog_burst_index.isna().mean(), 100*test.eog_burst_index.isna().mean()))
train.head()

train (9000, 23) | test (5000, 22)
eog_burst_index missing: train 50%, test 50%


,id,eeg_delta_power,eeg_theta_power,eeg_alpha_power,eeg_sigma_power,eeg_beta_power,eeg_gamma_power,eeg_slow_osc_power,eeg_spectral_entropy,eeg_spindle_density,...,eog_movement_density,eog_amplitude,heart_rate_mean,heart_rate_variability,respiration_rate,respiration_variability,spo2_mean,body_movement_index,eog_burst_index,sleep_stage
0,0,-1.51474,1.40728,10.33510,-1.61350,3.73081,0.99850,1.85689,-3.24253,-1.27096,...,2.65567,1.98733,1.60184,-2.49794,-0.59521,1.71154,1.93342,1.57365,-1.36230,1
1,1,-0.28998,0.89706,1.62494,2.41580,-2.70265,-0.10131,-1.68955,0.01442,-2.87943,...,4.36423,0.09942,3.38567,-0.56386,2.16016,-4.32301,1.07270,-2.43903,-0.37271,2
2,2,3.35435,0.32987,-5.41547,2.38680,-2.90584,-2.93372,-3.11713,-0.04647,1.61782,...,-3.87561,-0.87681,-2.84480,5.08383,-4.60411,0.37967,-2.06913,2.67324,NaN,3
3,3,-1.44917,-0.04374,1.71560,-1.27770,-0.19007,2.21826,1.69621,0.39756,0.00534,...,1.41415,0.39275,0.55060,-2.12910,2.32790,0.78319,0.98233,1.53824,-0.25040,1
4,4,1.35898,-2.36720,-7.40779,5.31815,-2.55954,-5.13284,-5.26634,1.73985,1.04618,...,-0.55616,0.86588,-1.96343,4.30036,0.22130,-1.44020,1.35760,-3.07701,-1.04947,3


## 5. Two imputers, one classifier
`enable_iterative_imputer` is required to import the (still experimental) `IterativeImputer`.

- **Median imputer** - fills missing `eog_burst_index` with a single constant (the column median). Simple, but throws away any relationship between this column and the others.
- **Iterative imputer** - models each feature with missing values as a regression on the OTHER features and predicts the missing entries. Because `eog_burst_index` is one of the more discriminative features (ANOVA F-score in the EDA) and is 50% missing, recovering plausible values per row - instead of one flat constant - hands the SVM real signal back.

In [2]:
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def build_svm(imputer, C=12, gamma=0.012):
    # impute -> scale -> RBF-SVM, as one leak-free estimator
    return make_pipeline(imputer, StandardScaler(),
                         SVC(kernel='rbf', C=C, gamma=gamma, random_state=42))

median_svm    = build_svm(SimpleImputer(strategy='median'),   C=10, gamma=0.01)   # previous best
iterative_svm = build_svm(IterativeImputer(max_iter=10, random_state=42), C=12, gamma=0.012)  # NEW best
iterative_svm

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('iterativeimputer', ...), ('standardscaler', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"random_state random_state: int, RandomState instance or None, default=NoneThe seed of the pseudo random number generator to use. Randomizesselection of estimator features if `n_nearest_features` is not `None`,the `imputation_order` if `random`, and the sampling from posterior if`sample_posterior=True`. Use an integer for determinism.See :term:`the Glossary <random_state>`.",42
,"estimator estimator: estimator object, default=BayesianRidge()The estimator to use at each step of the round-robin imputation.If `sample_posterior=True`, the estimator must support`return_std` in its `predict` method.",None
,"missing_values missing_values: int or np.nan, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`should be set to `np.nan`, since `pd.NA` will be converted to `np.nan`.",nan
,"sample_posterior sample_posterior: bool, default=FalseWhether to sample from the (Gaussian) predictive posterior of thefitted estimator for each imputation. Estimator must support`return_std` in its `predict` method if set to `True`. Set to`True` if using `IterativeImputer` for multiple imputations.",False
,"max_iter max_iter: int, default=10Maximum number of imputation rounds to perform before returning theimputations computed during the final round. A round is a singleimputation of each feature with missing values. The stopping criterionis met once `max(abs(X_t - X_{t-1}))/max(abs(X[known_vals])) < tol`,where `X_t` is `X` at iteration `t`. Note that early stopping is onlyapplied if `sample_posterior=False`.",10
,"tol tol: float, default=1e-3Tolerance of the stopping condition.",0.001
,"n_nearest_features n_nearest_features: int, default=NoneNumber of other features to use to estimate the missing values ofeach feature column. Nearness between features is measured usingthe absolute correlation coefficient between each feature pair (afterinitial imputation). To ensure coverage of features throughout theimputation process, the neighbor features are not necessarily nea

## 6. The key result: iterative imputation beats median imputation
We use **repeated** stratified CV (5 folds x 6 repeats = 30 scores) for a robust,
low-variance estimate, and a **paired** t-test because both models see the exact
same folds. A tiny p-value means the gain is real, not fold luck.

In [3]:
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from scipy import stats

rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=6, random_state=2024)
s_med = cross_val_score(median_svm,    train[FEATURES], y, cv=rcv, scoring='f1_macro', n_jobs=-1)
s_itr = cross_val_score(iterative_svm, train[FEATURES], y, cv=rcv, scoring='f1_macro', n_jobs=-1)

print('median-impute    : %.4f +/- %.4f' % (s_med.mean(), s_med.std()))
print('iterative-impute : %.4f +/- %.4f' % (s_itr.mean(), s_itr.std()))
diff = s_itr - s_med
print('mean paired gain : %+.4f   (better in %d/%d folds)' % (diff.mean(), (diff>0).sum(), len(diff)))
print('paired t-test p  : %.2e' % stats.ttest_rel(s_itr, s_med).pvalue)

median-impute    : 0.8323 +/- 0.0066
iterative-impute : 0.8420 +/- 0.0060
mean paired gain : +0.0097   (better in 29/30 folds)
paired t-test p  : 2.37e-11


## 7. Why we DON'T add the usual tricks
For the record (all tested, none beat the above by more than CV noise): polynomial/
interaction features and PCA-whitening HURT the SVM (the RBF kernel already models
interactions); a missing-indicator, KNN-imputation, bagging, NuSVC, poly-kernel,
LDA/QDA, ExtraTrees and stacking ensembles all scored at or below the baseline. The
data is near its ceiling for these inputs; the imputation was the one real lever.

## 8. Train final model on ALL data and write the submission
Refit the winning pipeline on all 9000 rows and predict the 5000 test rows.
The `IterativeImputer` learns its regression on train and applies it to test.

In [4]:
import os
os.makedirs(OUT_DIR, exist_ok=True)
final = build_svm(IterativeImputer(max_iter=10, random_state=42), C=12, gamma=0.012)
final.fit(train[FEATURES], y)
pred = final.predict(test[FEATURES])
submission = pd.DataFrame({'id': test['id'], 'sleep_stage': np.asarray(pred).astype(int).ravel()})
path = os.path.join(OUT_DIR, 'svm_iterimpute_submission.csv')
submission.to_csv(path, index=False)
print('wrote', path, submission.shape)
print('predicted class distribution:', dict(submission.sleep_stage.value_counts().sort_index()))
submission.head()

wrote ../outputs/svm_iterimpute_submission.csv (5000, 2)
predicted class distribution: {0: np.int64(1121), 1: np.int64(1290), 2: np.int64(1290), 3: np.int64(1299)}


,id,sleep_stage
0,9000,3
1,9001,3
2,9002,1
3,9003,3
4,9004,3


## 9. How to push further
- Calibrate probabilities (`CalibratedClassifierCV`) + per-class threshold tuning for the weak class 2.
- Try `IterativeImputer` with a stronger inner estimator (e.g. BayesianRidge -> a tree).
- Feed iterative-imputed features into the trees and re-test an ensemble (only the SVM benefited so far).